# v2.3.5 SUMA Remote Model Launcher: First Launch & Switching

Enable Colab GPU first, select model, MMProj, and context in the first form, then run.

To change models later, only execute the second form below. Both code cells are configured as Colab Forms with code collapsed by default; download progress and service outputs remain visible.

### Supported Models & Features

Preconfigured with 4 optimized public GGUF models. Context options: **64K**, **128K**, and **256K** (select according to allocated GPU memory to prevent OOM):

1. **Gemma 4 12B QAT Q4_0**
   - **Features**: Google's latest lightweight instruction-tuned model trained with Quantization-Aware Training (QAT). Low VRAM footprint with high precision; supports multimodal vision projector (MMProj).
   - **Multimodal (MMProj)**: Yes (built-in MMProj)
   - **Native Context**: Up to 256K tokens (Launcher supports up to 256K)

2. **Qwen3.5-9B Q5_K_M**
   - **Features**: Alibaba Qwen multilingual model with Q5_K_M precision. Balanced performance across comprehension and reasoning; also supports MMProj for vision tasks.
   - **Multimodal (MMProj)**: Yes (repository provides `mmproj-BF16.gguf` / `mmproj-F16.gguf`)
   - **Native Context**: Up to 262,144 tokens (256K, Launcher supports up to 256K)

3. **Qwen3.8-27B UD-Q2_K_XL**
   - **Features**: 27B parameter model compressed with extreme UD-Q2 quantization, enabling high-tier reasoning on 16GB GPUs (such as Colab T4); supports vision projector.
   - **Multimodal (MMProj)**: Yes (repository provides `mmproj-BF16.gguf` / `mmproj-F16.gguf`)
   - **Native Context**: Up to 262,144 tokens (256K, Launcher supports up to 256K)

4. **Inkling-2-Mini UD-Q2_K_XL**
   - **Features**: Thinking Machines' MoE multimodal model with Unsloth Dynamic 3.0 UD-Q2_K_XL quantization, balancing low VRAM usage, strong reasoning, and vision capabilities.
   - **Multimodal (MMProj)**: Yes (auto-fetches `mmproj-BF16.gguf`)
   - **Native Context**: Up to 256K tokens (Launcher supports up to 256K)

### Notes

- Uncensored options removed; only public open models retained. Downloads do not read/transmit HF_TOKEN.
- Official llama.cpp b12651 prebuilts with chunked download and resume support. No compilation or content hashing.
- Automatically fetches and launches SUMA Remote AI Bridge 0.3.6.0 Beta (with `--auto-approve` to automatically accept connections without manual verification). Live logs may contain sensitive pairing codes or tokens.
- Models stored under `/content/models` (independent of llama.cpp version). Cached models are reused across same-runtime switches.
- If selected model repository lacks MMProj, launcher automatically falls back to text-only mode.
- Helper commands available in additional cells: `show_suma_log()` or `stop_services()`.
- Execute forms individually as needed; 'Run All' will launch the first model then switch to the second.


In [ ]:
# @title Configure Model, MMProj & Context, Then Launch
MODEL = "Gemma 4 12B QAT Q4_0" # @param ["Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M", "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL"]
LOAD_MMPROJ = True # @param {type:"boolean"}
CONTEXT = "64K" # @param ["64K", "128K", "256K"]

import codecs
import json
import os
import platform
import re
import shlex
import shutil
import signal
import socket
import subprocess
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

_VERSION = 'b12651'
_ROOT = Path('/content/llama-direct-b12651') if Path('/content').exists() else Path.home()/'llama-direct-b12651'
_MODELS_ROOT = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
_KNOWN_ASSETS = {
    'b12651': [
        {'name':'llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz','size':220195495,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz','size':594371475,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz'},
        {'name':'llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz','size':224478514,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz','size':444078776,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz'},
        {'name':'llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz','size':219392133,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz','size':559873124,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz'}
    ]
}
_CTX = {'64K':65536,'128K':131072,'256K':262144}[CONTEXT]
_PORT = 8081
_TIMEOUT = 900
_NETWORK_TIMEOUT = 45
_RETRIES = 3
_MODELS = {
    'Gemma 4 12B QAT Q4_0': {'repo':'google/gemma-4-12B-it-qat-q4_0-gguf','file':'gemma-4-12b-it-qat-q4_0.gguf','alias':'gemma-4-12b','mmproj':'mmproj-gemma-4-12b-it-qat-q4_0.gguf'},
    'Qwen3.5-9B Q5_K_M': {'repo':'unsloth/Qwen3.5-9B-GGUF','file':'Qwen3.5-9B-Q5_K_M.gguf','alias':'qwen3.5-9b'},
    'Qwen3.8-27B UD-Q2_K_XL': {'repo':'unsloth/Qwen3.8-27B-GGUF','file':'Qwen3.8-27B-UD-Q2_K_XL.gguf','alias':'qwen3.8-27b-q2kxl'},
    'Inkling-2-Mini UD-Q2_K_XL': {'repo':'unsloth/Inkling-2-Mini-GGUF','file':'Mainline_Support/Inkling-2-Mini-UD-Q2_K_XL.gguf_file','alias':'inkling-2-mini-q2kxl','mmproj':'mmproj-BF16.gguf'},
}
_SELECTED = _MODELS[MODEL]

def _secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass
    return None

def _human(value):
    if value is None:
        return 'Unknown'
    value = float(value)
    for unit in ('B','KiB','MiB','GiB','TiB'):
        if abs(value)<1024 or unit=='TiB':
            return f'{value:.2f} {unit}'
        value /= 1024

def _clock(seconds):
    if seconds is None:
        return '--:--'
    seconds=max(0,int(seconds))
    return f'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}'

class _Progress:
    def __init__(self,label,total=None,initial=0):
        self.label,self.total,self.done=label,total,initial
        self.initial=initial
        self.phase='Connecting / Waiting for response'
        self.started=self.last_data=time.monotonic()
        self.event=threading.Event();self.lock=threading.Lock()
    def __enter__(self):
        print('\n▶ '+self.label,flush=True);self.render()
        self.thread=threading.Thread(target=self.tick,daemon=True);self.thread.start()
        return self
    def configure(self,total=None,initial=0,phase=None):
        if phase is None:
            phase='Downloading'
        with self.lock:
            self.total,self.done,self.initial=total,initial,initial
            self.phase=phase;self.started=self.last_data=time.monotonic()
    def update(self,count):
        with self.lock:
            self.done+=count;self.last_data=time.monotonic()
    def render(self,final=None):
        with self.lock:
            now=time.monotonic();elapsed=now-self.started
            speed=max(0,self.done-self.initial)/max(elapsed,0.001)
            percentage=f'{min(100,self.done/self.total*100):6.2f}%' if self.total else '   --%'
            eta=max(0,self.total-self.done)/speed if self.total and speed>0 else None
            phase=self.phase
            if phase == 'Downloading' and now-self.last_data>=5:
                phase=f'Waiting for data {int(now - self.last_data)}s'
            elapsed_label = 'Elapsed'
            line=f'{percentage} | {_human(self.done)} / {_human(self.total)} | {_human(speed)}/s | {elapsed_label} {_clock(elapsed)} | ETA {_clock(eta)} | {final or phase}'
            print('\r'+line.ljust(155),end='',flush=True)
    def tick(self):
        while not self.event.wait(1):
            self.render()
    def __exit__(self,kind,value,traceback):
        self.event.set();self.thread.join(timeout=2)
        self.render('Done' if kind is None else 'Interrupted / Failed');print(flush=True)

class _SafeRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self,request,fp,status,message,headers,newurl):
        redirected=super().redirect_request(request,fp,status,message,headers,newurl)
        if redirected and urllib.parse.urlparse(request.full_url).hostname!=urllib.parse.urlparse(newurl).hostname:
            redirected.remove_header('Authorization')
        return redirected

_OPENER=urllib.request.build_opener(_SafeRedirect())

def _request_url(url):
    if urllib.parse.urlsplit(url).hostname!='huggingface.co':return url
    parts=urllib.parse.urlsplit(url)
    query=urllib.parse.parse_qsl(parts.query,keep_blank_values=True)
    query=[(key,value) for key,value in query if key!='_colab_retry']
    query.append(('_colab_retry',str(time.time_ns())))
    return urllib.parse.urlunsplit(parts._replace(query=urllib.parse.urlencode(query)))

def _error(exc):
    if isinstance(exc,urllib.error.HTTPError):
        host=urllib.parse.urlparse(str(exc.url)).hostname or ''
        if 'huggingface.co' in host and exc.code in (401,403):
            return f'HTTP {exc.code}: Anonymous download failed. Gated/private models not supported.'
        if host=='api.github.com' and exc.code in (403,429):
            return 'GitHub API rate limit or permission error; please set GITHUB_TOKEN and retry.'
        return f'HTTP {exc.code}：{exc.reason}'
    return str(exc)

def _read_json(url,label,token=None):
    headers={'User-Agent':'colab-simple-model-runner','Accept':'application/json','Cache-Control':'no-cache'}
    if token:headers['Authorization']='Bearer '+token
    for attempt in range(_RETRIES):
        try:
            with _Progress(label) as progress:
                with _OPENER.open(urllib.request.Request(_request_url(url),headers=headers),timeout=_NETWORK_TIMEOUT) as response:
                    length=response.headers.get('Content-Length');progress.configure(int(length) if length else None)
                    data=bytearray()
                    while True:
                        block=response.read1(64*1024)
                        if not block:break
                        data.extend(block);progress.update(len(block))
            return json.loads(data)
        except (urllib.error.URLError,TimeoutError,OSError) as exc:
            if (isinstance(exc,urllib.error.HTTPError) and exc.code in (400,401,403,404)) or attempt==_RETRIES-1:
                raise RuntimeError(_error(exc)) from exc
            print(f'{_error(exc)}; Retry {attempt + 2}/{_RETRIES}',flush=True);time.sleep(2**attempt)

def _size_matches(path,size=None):
    path=Path(path)
    return path.is_file() and path.stat().st_size>0 and (size is None or path.stat().st_size==size)

def _download(url,dest,label=None,size=None,token=None):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if _size_matches(dest,size):
        print(f'✓ Reusing {dest.name} ({_human(dest.stat().st_size)}; hash check skipped)',flush=True);return dest
    partial=dest.with_name(dest.name+'.part');meta=dest.with_name(dest.name+'.part.json')
    try:saved=json.loads(meta.read_text()) if meta.exists() else {}
    except (ValueError,OSError):saved={}
    if saved.get('url')!=url or (size is not None and partial.exists() and partial.stat().st_size>size):
        partial.unlink(missing_ok=True);saved={}
    for attempt in range(_RETRIES):
        offset=partial.stat().st_size if partial.exists() else 0
        if offset and size is not None and offset==size:
            partial.replace(dest);meta.unlink(missing_ok=True)
            print('✓ Download complete: ',dest.name,flush=True);return dest
        headers={'User-Agent':'colab-simple-model-runner','Accept-Encoding':'identity','Cache-Control':'no-cache'}
        if token:headers['Authorization']='Bearer '+token
        if offset:
            headers['Range']=f'bytes={offset}-'
            if saved.get('validator'):headers['If-Range']=saved['validator']
        print('Download source: ',url,flush=True)
        try:
            with _Progress(label or dest.name,size,offset) as progress:
                with _OPENER.open(urllib.request.Request(_request_url(url),headers=headers),timeout=_NETWORK_TIMEOUT) as response:
                    if getattr(response,'status',200)==206:
                        match=re.fullmatch(r'bytes (\d+)-(\d+)/(\d+|\*)',response.headers.get('Content-Range',''))
                        if not match or int(match.group(1))!=offset:
                            partial.unlink(missing_ok=True);raise RuntimeError('Resume byte range error; temporary cache removed.')
                        total=int(match.group(3)) if match.group(3)!='*' else size
                    else:
                        if offset:print('\nServer did not accept resume range or source was modified; restarting from beginning.',flush=True)
                        offset=0;length=response.headers.get('Content-Length');total=int(length) if length else size
                    if size is not None and total is not None and size!=total:
                        partial.unlink(missing_ok=True);raise RuntimeError('Response content length mismatch.')
                    validator=response.headers.get('ETag') or response.headers.get('Last-Modified')
                    if validator and validator.startswith('W/'):validator=response.headers.get('Last-Modified')
                    saved={'url':url,'validator':validator};meta.write_text(json.dumps(saved))
                    progress.configure(total,offset)
                    with partial.open('ab' if offset else 'wb') as output:
                        while True:
                            block=response.read1(1024*1024)
                            if not block:break
                            output.write(block);progress.update(len(block))
                    if total is not None and partial.stat().st_size!=total:
                        raise RuntimeError(f'Download incomplete: {partial.stat().st_size}/{total} bytes')
            if not _size_matches(partial,size):raise RuntimeError('Downloaded file is empty or size does not match.')
            partial.replace(dest);meta.unlink(missing_ok=True)
            print(f'✓ Download complete: {dest.name} ({_human(dest.stat().st_size)})',flush=True);return dest
        except (urllib.error.URLError,TimeoutError,OSError,RuntimeError) as exc:
            if isinstance(exc,urllib.error.HTTPError):
                if exc.code in (400,401,403,404):raise RuntimeError(_error(exc)) from exc
                if exc.code==416:partial.unlink(missing_ok=True);meta.unlink(missing_ok=True);saved={}
            if attempt==_RETRIES-1:raise RuntimeError(_error(exc)) from exc
            print(f'{_error(exc)}; Retry {attempt + 2}/{_RETRIES}',flush=True);time.sleep(2**attempt)

def _extract(archive,dest):
    dest=Path(dest);marker=dest/'.extracted'
    if marker.exists():print('✓ Already extracted: ',dest.name,flush=True);return
    if not hasattr(tarfile,'data_filter'):raise RuntimeError('Please update your Python / Colab runtime to support tarfile data_filter.')
    dest.mkdir(parents=True,exist_ok=True)
    with tarfile.open(archive,'r:gz') as tar:
        with _Progress('Reading archive index: '+Path(archive).name):members=tar.getmembers()
        total=sum(item.size for item in members if item.isfile())
        with _Progress('Extracting: '+Path(archive).name,total) as progress:
            progress.configure(total,phase='Extracting')
            for item in members:
                tar.extract(item,dest,filter='data')
                if item.isfile():progress.update(item.size)
    marker.touch();print('✓ Extraction complete: ',dest,flush=True)

def _capture(cmd,env=None):
    result=subprocess.run(list(map(str,cmd)),capture_output=True,text=True,errors='replace',env=env,timeout=60)
    if result.returncode:raise RuntimeError(f'Exit code {result.returncode}\n{result.stdout}\n{result.stderr}')
    return result.stdout+result.stderr

def _env(paths):
    env={key:value for key,value in os.environ.items() if not key.startswith('LLAMA_ARG_') and key not in ('LLAMA_API_KEY','LLAMA24_API_KEY')}
    env['LD_LIBRARY_PATH']=':'.join(dict.fromkeys([*map(str,paths),*filter(None,env.get('LD_LIBRARY_PATH','').split(':'))]))
    key=os.environ.get('LLAMA24_API_KEY','')
    if key:env['LLAMA_API_KEY']=key
    return env

def _candidates(assets,arch,driver):
    candidates=[]
    fallback=[]
    for asset in assets:
        match=re.fullmatch(rf'llama-{re.escape(_VERSION)}-bin-ubuntu-cuda-(\d+)\.(\d+)-{re.escape(arch)}\.tar\.gz',asset['name'])
        if not match:continue
        cuda=tuple(map(int,match.groups()))
        if cuda<=driver:
            candidates.append((cuda,asset))
        elif cuda[0]==driver[0]:
            fallback.append((cuda,asset))
        else:
            print(f"Skipping {asset['name']}: Higher than host CUDA driver {driver[0]}.{driver[1]}",flush=True)
    if not candidates and fallback:
        fallback.sort(key=lambda pair:pair[0])
        print(f"No matching <= {driver[0]}.{driver[1]} package found, trying same major version: {fallback[0][1]['name']}",flush=True)
        return [fallback[0][1]]
    return [asset for _,asset in sorted(candidates,key=lambda pair:pair[0],reverse=True)]

def _generate_candidate_assets(version,arch):
    base=f'https://github.com/ggml-org/llama.cpp/releases/download/{version}'
    items=[]
    for cver in ('13.5','12.8','12.4','12.2','12.0','11.8'):
        items.append({'name':f'llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','browser_download_url':f'{base}/llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','size':None})
        items.append({'name':f'cudart-llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','browser_download_url':f'{base}/cudart-llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','size':None})
    return items

def _test_binary(info):
    exe=Path(info['server']);env=_env(info['library_paths'])
    if not exe.is_file():raise RuntimeError('Executable binary missing.')
    print(_capture([exe,'--version'],env))
    if shutil.which('ldd'):
        targets=[exe,*exe.parent.glob('libggml-cuda.so*')]
        for target in targets:
            deps=_capture(['ldd',target],env)
            if 'not found' in deps:raise RuntimeError(deps)
    devices=_capture([exe,'--list-devices'],env);print(devices)
    if not re.search(r'\bCUDA\d+\s*:',devices):raise RuntimeError('No available CUDA devices detected.')
    return info

def _ensure_llama():
    if sys.platform!='linux' or not shutil.which('nvidia-smi'):
        raise RuntimeError('Please enable GPU hardware accelerator in Colab runtime.')
    smi=_capture(['nvidia-smi']);print(smi)
    match=re.search(r'CUDA Version:\s*(\d+)\.(\d+)',smi)
    if not match:raise RuntimeError('Cannot parse host CUDA driver version.')
    driver=tuple(map(int,match.groups()));record=_ROOT/'installation.json'
    if record.exists():
        try:
            info=json.loads(record.read_text());version=re.search(r'-cuda-(\d+)\.(\d+)-',info.get('asset',''))
            if info.get('tag')==_VERSION and version and tuple(map(int,version.groups()))<=driver:
                print('▶ Reusing already downloaded llama.cpp.',flush=True)
                return _test_binary(info)
        except (RuntimeError,OSError,ValueError,KeyError,subprocess.TimeoutExpired) as exc:
            print('Existing installation failed validation, selecting prebuilt package: ',exc,flush=True)
    arch={'x86_64':'x64','aarch64':'arm64'}.get(platform.machine())
    if not arch:raise RuntimeError('Unsupported CPU architecture.')
    assets=_KNOWN_ASSETS.get(_VERSION)
    if not assets:
        try:
            release=_read_json(f'https://api.github.com/repos/ggml-org/llama.cpp/releases/tags/{_VERSION}','Fetching official release assets',_secret('GITHUB_TOKEN'))
            if release.get('tag_name')==_VERSION and not release.get('draft'):
                assets=release.get('assets',[])
        except Exception as exc:
            print(f'Failed to fetch GitHub API release list ({exc}), falling back to direct release download list...',flush=True)
            assets=_generate_candidate_assets(_VERSION,arch)
    if not assets:
        assets=_generate_candidate_assets(_VERSION,arch)
    by_name={asset['name']:asset for asset in assets}
    candidates=_candidates(assets,arch,driver)
    if not candidates:raise RuntimeError('No matching official CUDA prebuilts found; compilation is disabled.\n'+'\n'.join(by_name))
    def fetch(asset):
        url=asset['browser_download_url']
        if Path(asset['name']).name!=asset['name'] or not url.startswith(f'https://github.com/ggml-org/llama.cpp/releases/download/{_VERSION}/'):
            raise ValueError('Asset is not from the specified official release.')
        return _download(url,_ROOT/'downloads'/asset['name'],asset['name'],asset.get('size'))
    errors=[]
    for asset in candidates:
        try:
            package=_ROOT/'packages'/asset['name'].removesuffix('.tar.gz');_extract(fetch(asset),package)
            runtime='cudart-'+asset['name']
            if runtime in by_name:_extract(fetch(by_name[runtime]),package/'cuda-runtime')
            servers=list(package.rglob('llama-server'))
            if len(servers)!=1:raise RuntimeError(f'Expected exactly 1 llama-server executable, found {len(servers)}.')
            exe=servers[0].resolve();exe.chmod(exe.stat().st_mode|0o111)
            libraries=sorted({str(path.parent.resolve()) for path in package.rglob('*.so*') if path.is_file()})
            info={'tag':_VERSION,'asset':asset['name'],'server':str(exe),'library_paths':list(dict.fromkeys([str(exe.parent),*libraries])),'method':'official-prebuilt-download'}
            _test_binary(info);record.write_text(json.dumps(info,ensure_ascii=False,indent=2));return info
        except (RuntimeError,OSError,tarfile.TarError,subprocess.TimeoutExpired) as exc:
            errors.append(f'{asset["name"]}: {exc}');print('Package unusable: ',exc,flush=True)
    raise RuntimeError('\n\n'.join(errors))

def _gguf(path,main=False):
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    with path.open('rb') as source:
        if source.read(4)!=b'GGUF':raise RuntimeError('Not a valid GGUF file: '+str(path))
    if main:
        match=re.search(r'-(\d{5})-of-(\d{5})\.gguf$',path.name)
        if match:
            index,count=map(int,match.groups())
            if index!=1:raise RuntimeError('Split models must start with part 00001.')
            for number in range(1,count+1):
                name=re.sub(r'-\d{5}-of-\d{5}\.gguf$',f'-{number:05d}-of-{count:05d}.gguf',path.name)
                if not path.with_name(name).is_file():raise RuntimeError('Missing split part: '+name)
    print(f'✓ File ready: {path.name} ({_human(path.stat().st_size)}; hash check skipped)',flush=True);return path

def _projector(names, required=True):
    candidates=[name for name in names if name.lower().endswith('.gguf') and ('mmproj' in Path(name).name.lower() or 'vision' in Path(name).name.lower()) and not any(word in Path(name).name.lower() for word in ('assistant','mtp','imatrix'))]
    if not candidates:
        if required:raise RuntimeError('No recognizable MMProj projector in repo; uncheck LOAD_MMPROJ and retry.')
        return None
    candidates.sort(key=lambda name:('mmproj' not in name.lower(),not any(word in name.lower() for word in ('f16','fp16')),name))
    print('Auto-selected MMProj: ',candidates[0],flush=True);return candidates[0]

def _prepare_files():
    repo=_SELECTED['repo'];folder=_MODELS_ROOT/repo.replace('/','--')
    result={};pending=[];model_name=_SELECTED['file']
    local_model=model_name[:-5] if model_name.endswith('.gguf_file') else model_name
    if (folder/local_model).is_file():result['model']=_gguf(folder/local_model,True)
    else:pending.append(('model',model_name))
    metadata=_ROOT/'metadata'/(repo.replace('/','--')+'.json')
    if LOAD_MMPROJ:
        projector_name=_SELECTED.get('mmproj')
        if not projector_name and folder.exists():
            names=[str(path.relative_to(folder)) for path in folder.rglob('*.gguf')]
            projector_name=_projector(names, required=False)
        if not projector_name and metadata.exists():
            try:
                cached=json.loads(metadata.read_text());projector_name=_projector([item['rfilename'] for item in cached.get('siblings',[])], required=False)
            except (ValueError,KeyError,OSError):pass
        if projector_name:
            if (folder/projector_name).is_file():result['mmproj']=_gguf(folder/projector_name)
            else:pending.append(('mmproj',projector_name))
        else:
            pending.append(('mmproj',None))
    if not pending:
        print('✓ Using cached local files; skipping remote API checks and rescans.',flush=True);return result
    token=None;encoded=urllib.parse.quote(repo,safe='/')
    info=_read_json(f'https://huggingface.co/api/models/{encoded}?blobs=true','Fetching repository file list',token)
    if info.get('private') or info.get('gated') not in (None,False):
        raise RuntimeError('Repository requires authorization/gated access; this version only runs public models.')
    metadata.parent.mkdir(parents=True,exist_ok=True)
    metadata.write_text(json.dumps(info,ensure_ascii=False));revision=info.get('sha')
    if not revision:raise RuntimeError('HF API did not return commit SHA; cannot pin download revision.')
    files={item['rfilename']:item for item in info.get('siblings',[])}
    for kind,name in pending:
        if kind=='mmproj' and not name:
            name=_projector(list(files), required=False)
            if not name:
                print('ℹ No MMProj projector in repository; automatically switched to text-only mode.',flush=True)
                continue
        if name not in files:raise RuntimeError(f'Repository missing {name}.\nAvailable GGUF:\n'+'\n'.join(key for key in files if key.endswith('.gguf')))
        local_name=name[:-5] if name.endswith('.gguf_file') else name
        path=folder/local_name;path.parent.mkdir(parents=True,exist_ok=True)
        item=files[name];size=item.get('size') or (item.get('lfs') or {}).get('size')
        url=f'https://huggingface.co/{encoded}/resolve/{revision}/{urllib.parse.quote(name,safe="/")}?download=true'
        if size and not _size_matches(path,size):
            partial=path.with_name(path.name+'.part');meta=path.with_name(path.name+'.part.json')
            try:saved=json.loads(meta.read_text()) if meta.exists() else {}
            except (ValueError,OSError):saved={}
            reusable=partial.stat().st_size if partial.exists() and saved.get('url')==url else 0
            if shutil.disk_usage(path.parent).free<max(0,size-reusable)+1024**3:raise RuntimeError('Insufficient disk space: '+name)
        _download(url,path,('Model: ' if kind=='model' else 'MMProj: ')+name,size,token)
        result[kind]=_gguf(path,kind=='model')
    return result

def _identity(pid):
    try:
        proc=Path('/proc')/str(pid);fields=(proc/'stat').read_text().rsplit(')',1)[1].split()
        return {'pid':pid,'start_ticks':fields[19],'exe':str((proc/'exe').resolve(strict=True))}
    except (FileNotFoundError,ProcessLookupError,PermissionError):return None

def _stop_owned(name='server'):
    path=_ROOT/(name+'-process.json')
    if not path.exists():return
    try:saved=json.loads(path.read_text())
    except Exception:saved={}
    pid=saved.get('pid')
    if pid and _identity(pid)==saved:
        try:
            os.kill(pid,signal.SIGTERM)
            for _ in range(50):
                if _identity(pid)!=saved:break
                time.sleep(0.2)
            if _identity(pid)==saved:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    try:
        if pid:os.waitpid(pid,os.WNOHANG)
    except (ChildProcessError,OSError):pass
    path.unlink(missing_ok=True)

def _tail(path,limit=14000):
    path=Path(path)
    if not path.exists():return '(No log yet)'
    with path.open('rb') as source:
        source.seek(0,2);source.seek(max(0,source.tell()-limit));return source.read().decode('utf-8',errors='replace')

def _health(path='/health'):
    key=os.environ.get('LLAMA24_API_KEY','');headers={'Authorization':'Bearer '+key} if key else {}
    request=urllib.request.Request(f'http://127.0.0.1:{_PORT}{path}',headers=headers)
    opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open(request,timeout=5) as response:return json.load(response)

def _local_model_path(item):
    name = item['file']
    local_name = name[:-5] if name.endswith('.gguf_file') else name
    return _MODELS_ROOT/item['repo'].replace('/', '--')/local_name

def _sync_models_dir():
    models_dir = _ROOT/'active_models'
    models_dir.mkdir(parents=True, exist_ok=True)
    for name, item in _MODELS.items():
        m_path = _local_model_path(item)
        link_path = models_dir/f"{item['alias']}.gguf"
        if m_path.is_file():
            if not link_path.exists():
                try:
                    link_path.symlink_to(m_path)
                except (OSError, NotImplementedError):
                    try:
                        os.link(m_path, link_path)
                    except OSError:
                        shutil.copyfile(m_path, link_path)
        else:
            if link_path.is_symlink() or link_path.is_file():
                link_path.unlink(missing_ok=True)
    return models_dir

def _model_section(item,paths,models_dir=None):
    # Build one router preset section for a downloaded model; None when not downloaded.
    folder=_MODELS_ROOT/item['repo'].replace('/','--')
    selected=item['alias']==_SELECTED['alias']
    local_model=paths['model'] if selected else _local_model_path(item)
    if not Path(local_model).is_file():return None
    unified=(models_dir or (_ROOT/'active_models'))/f"{item['alias']}.gguf"
    model=unified if unified.is_file() else local_model
    projector=None
    if LOAD_MMPROJ:
        if selected:projector=paths.get('mmproj')
        else:
            name=item.get('mmproj') or (_projector([str(p.relative_to(folder)) for p in folder.rglob('*.gguf')],required=False) if folder.exists() else None)
            if name and (folder/name).is_file():projector=folder/name
    micro=256 if projector else (64 if _CTX>=131072 else 128)
    lines=[f"[{item['alias']}]",f'model = {model}',f'ubatch-size = {micro}']
    if projector:lines.append(f'mmproj = {projector}')
    return lines+['']

def _write_preset(paths):
    # Router mode lists every downloaded model in the unified active_models dir; only one stays loaded (T4 VRAM).
    models_dir=_sync_models_dir()
    threads=max(1,min(8,os.cpu_count() or 2))
    lines=['[*]',f'ctx-size = {_CTX}','parallel = 1',f'threads = {threads}','batch-size = 512',
           'flash-attn = on','cache-type-k = q8_0','cache-type-v = q8_0','fit = on','fit-target = 1024','cache-ram = 0','']
    for item in _MODELS.values():
        section=_model_section(item,paths,models_dir)
        if section:lines+=section
    preset=_ROOT/'models-preset.ini';preset.write_text('\n'.join(lines),encoding='utf-8')
    return preset

def _command(exe,paths,help_text=''):
    if '--models-preset' in help_text:
        return [str(exe),'--models-preset',str(_write_preset(paths)),'--models-max','1','--host','127.0.0.1','--port',str(_PORT)]
    # Fallback for builds without router mode: single model only.
    has_mmproj = LOAD_MMPROJ and 'mmproj' in paths
    micro=256 if has_mmproj else (64 if _CTX>=131072 else 128)
    threads=max(1,min(8,os.cpu_count() or 2))
    cmd=[str(exe),'--model',str(paths['model']),'--alias',_SELECTED['alias'],'--host','127.0.0.1','--port',str(_PORT),
         '--ctx-size',str(_CTX),'--parallel','1','--threads',str(threads),'--batch-size','512','--ubatch-size',str(micro),
         '--flash-attn','on','--cache-type-k','q8_0','--cache-type-v','q8_0','--fit','on','--fit-target','1024','--cache-ram','0']
    cmd+=['--mmproj',str(paths['mmproj'])] if has_mmproj else ['--no-mmproj']
    return cmd

def _router_ready():
    # Ask the router to load the selected model and report whether it is ready.
    alias=_SELECTED['alias']
    status={m.get('id'):(m.get('status') or {}).get('value') for m in _health('/v1/models').get('data',[])}
    current=status.get(alias)
    if current is None:raise RuntimeError(f'Router did not list model: {alias}')
    if current=='loaded':return True
    if current not in ('loading',):
        if current not in ('unloaded',None):print(f'Model status: {current}; requesting reload.',flush=True)
        key=os.environ.get('LLAMA24_API_KEY','')
        headers={'Content-Type':'application/json',**({'Authorization':'Bearer '+key} if key else {})}
        request=urllib.request.Request(f'http://127.0.0.1:{_PORT}/models/load',data=json.dumps({'model':alias}).encode(),headers=headers)
        opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
        with opener.open(request,timeout=10) as response:response.read()
    return False

def _start_model(info,paths):
    exe=Path(info['server']);env=_env(info['library_paths'])
    help_text=_capture([exe,'--help'],env)
    cmd=_command(exe,paths,help_text)
    missing=[arg for arg in cmd[1:] if arg.startswith('--') and arg not in help_text]
    if missing:raise RuntimeError(f'Binary does not support arguments: {missing}')
    bound=False
    for _ in range(25):
        with socket.socket(socket.AF_INET,socket.SOCK_STREAM) as sock:
            try:
                sock.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
                sock.bind(('127.0.0.1',_PORT))
                bound=True;break
            except OSError:
                time.sleep(0.2)
    if not bound:
        raise RuntimeError(f'Port {_PORT} is unavailable or already bound; please retry shortly.')
    log=_ROOT/'logs'/f'llama-server-{time.time_ns()}.log';(_ROOT/'last-log.txt').write_text(str(log))
    print('▶ Loading model and warming up; not downloading or hashing.',flush=True);print('Executing: ',shlex.join(cmd),flush=True)
    with log.open('wb') as output:
        process=subprocess.Popen(cmd,cwd=str(exe.parent),env=env,stdin=subprocess.DEVNULL,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    saved=_identity(process.pid)
    if saved:(_ROOT/'server-process.json').write_text(json.dumps(saved))
    try:
        deadline=time.monotonic()+_TIMEOUT;report=0
        while time.monotonic()<deadline:
            if process.poll() is not None:
                print(_tail(log));raise RuntimeError(f'llama-server exited with code {process.returncode}; Log: {log}')
            try:
                if _health().get('status')=='ok' and process.poll() is None and ('--models-preset' not in cmd or _router_ready()):
                    models=_health('/v1/models');print('✓ Model service ready: ',models,flush=True)
                    print(f"API: http://127.0.0.1:{_PORT}/v1\nModel alias: {_SELECTED['alias']}",flush=True);return process
            except (urllib.error.URLError,OSError,TimeoutError,ValueError):pass
            if time.monotonic()-report>=15:
                print('Model loading / warming up...\n'+_tail(log,2000),flush=True);report=time.monotonic()
            time.sleep(2)
        print(_tail(log));raise TimeoutError(f'Timeout waiting for model ({_TIMEOUT}s); Log: {log}')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        (_ROOT/'server-process.json').unlink(missing_ok=True);raise

def _follow_suma(log,process,event,state):
    decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        with Path(log).open('rb') as source:
            while not event.is_set():
                block=source.read(8192)
                if block:
                    state['bytes']+=len(block);text=decoder.decode(block)
                    if text:print(text,end='',flush=True)
                    continue
                if process.poll() is not None:break
                event.wait(0.2)
        text=decoder.decode(b'',final=True)
        if text:print(text,end='',flush=True)
    except Exception as exc:print(f'\n[SUMA] Failed to sync log: {exc}; Log: {log}',flush=True)

def _stop_watcher():
    event=globals().get('_suma_watch_stop')
    if event is not None:event.set()
    thread=globals().get('_suma_watch_thread')
    if thread is not None and thread.is_alive():thread.join(timeout=2)

def _start_suma():
    global suma_process,_suma_watch_stop,_suma_watch_thread
    if _health().get('status')!='ok':raise RuntimeError('Model not ready, cannot start SUMA.')
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:raise RuntimeError('Unsupported CPU architecture for SUMA.')
    work=_ROOT/'suma';work.mkdir(parents=True,exist_ok=True)
    tag='0360';name=f'SUMA-Remote-AI-Bridge_0.3.6.0_Beta-linux-{arch}.tar.gz'
    url=f'https://github.com/marchfun1/SUMA-Remote-AI-Bridge/releases/download/{tag}/{name}'
    size=8190363 if arch=='amd64' else None
    archive=_download(url,_ROOT/'downloads'/name,f'SUMA AI Bridge ({tag})',size)
    package=work/f'package-{tag}-{arch}';_extract(archive,package)
    binaries=[path for path in package.rglob('suma-ai-bridge') if path.is_file()]
    if len(binaries)!=1:raise RuntimeError('Could not locate unique SUMA executable.')
    binary=binaries[0].resolve();binary.chmod(binary.stat().st_mode|0o111)
    _stop_watcher();_stop_owned('suma');log=work/'suma-ai-bridge.log'
    if log.exists() and log.stat().st_size:log.rename(work/f'suma-ai-bridge-{time.time_ns()}.log')
    print('\n[SUMA] Auto-started; the following live output may contain pairing codes or tokens, keep confidential.',flush=True)
    with log.open('wb') as output:
        suma_process=subprocess.Popen([str(binary), '--port', str(_PORT), '--auto-approve'], cwd=str(work), stdin=subprocess.DEVNULL, stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
    saved=_identity(suma_process.pid)
    if saved:(_ROOT/'suma-process.json').write_text(json.dumps(saved))
    _suma_watch_stop=threading.Event();state={'bytes':0}
    _suma_watch_thread=threading.Thread(target=_follow_suma,args=(log,suma_process,_suma_watch_stop,state),daemon=True);_suma_watch_thread.start()
    deadline=time.monotonic()+5
    while time.monotonic()<deadline and suma_process.poll() is None:time.sleep(0.2)
    if suma_process.poll() is not None:
        _suma_watch_thread.join(timeout=2);raise RuntimeError(f'SUMA exited with code {suma_process.returncode}; Log: {log}')
    print(f'\n[SUMA] Process running with PID {suma_process.pid}.',flush=True)
    if not state['bytes']:print('[SUMA] No output received yet.',flush=True)
    print('[SUMA] Process alive does not guarantee pairing / connection success; refer to actual logs.',flush=True)
    return suma_process

def list_downloaded_models():
    print('=== Downloaded Models Ready For Switching ===', flush=True)
    found = 0
    for name, item in _MODELS.items():
        m_path = _local_model_path(item)
        if m_path.is_file():
            found += 1
            curr = ' [Currently Active]' if name == MODEL else ''
            print(f'✓ {name} ({_human(m_path.stat().st_size)}){curr}', flush=True)
        else:
            print(f'  {name} (Not downloaded)', flush=True)
    if not found:
        print('(No models downloaded yet)', flush=True)
    print('=====================================', flush=True)

def show_suma_log():
    print(_tail(_ROOT/'suma'/'suma-ai-bridge.log',20000),flush=True)

def _is_suma_alive():
    path=_ROOT/'suma-process.json'
    if not path.exists():return False
    try:saved=json.loads(path.read_text())
    except Exception:return False
    pid=saved.get('pid')
    return bool(pid and _identity(pid)==saved)

def stop_model_service():
    _stop_owned('server')
    print('Stopped previous llama-server; SUMA AI Bridge remains running.',flush=True)

def stop_services():
    _stop_watcher();_stop_owned('suma');_stop_owned()
    print('Stopped tracked processes matching identity.',flush=True)

def _run():
    for path in (_ROOT,_ROOT/'downloads',_ROOT/'logs',_ROOT/'metadata'):path.mkdir(parents=True,exist_ok=True)
    print(f'Model: {MODEL}\nLoad MMProj: {LOAD_MMPROJ}\nContext: {_CTX}\nSUMA: Auto-enabled\nHash verification: Disabled',flush=True)
    _stop_watcher();_stop_owned('suma');_stop_owned()
    info=_ensure_llama();paths=_prepare_files()
    process=_start_model(info,paths)
    try:_start_suma()
    except Exception:
        print('[SUMA] Launch failed; model server remains running. Error details:',flush=True);raise
    return process

server_process = _run()


## Model Switching

This form provides three options: New Model, Load MMProj, and Context (64K / 128K / 256K). Select options and run.
Switch models anytime using the form below without restarting SUMA AI Bridge.

- Previously downloaded models are **not deleted**; newly downloaded models are placed in the same unified directory (`active_models`) so `llama.cpp` can switch between all downloaded models at any time.
- Once downloaded and prepared, only the previous model is stopped and the new model is loaded; SUMA AI Bridge remains running without re-pairing.
- If loading fails, the previous service has stopped but model files remain cached; re-run with the previous model or a smaller context.


In [ ]:
# @title Switch Model
NEW_MODEL = "Qwen3.8-27B UD-Q2_K_XL" # @param ["Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M", "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL"]
MMPROJ = True # @param {type:"boolean"}
CONTEXT_SIZE = "64K" # @param ["64K", "128K", "256K"]

def download_and_switch(model, mmproj, context):
    global MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED, server_process
    required = ("_MODELS", "_ROOT", "_ensure_llama", "_prepare_files",
                "_start_model", "_start_suma", "stop_services", "stop_model_service", "_sync_models_dir", "list_downloaded_models", "MODEL",
                "LOAD_MMPROJ", "CONTEXT", "_CTX", "_SELECTED")
    if any(name not in globals() for name in required):
        raise RuntimeError('Please run the main setup cell above first in the same runtime.')
    allowed = ("Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M",
               "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL")
    contexts = {"64K":65536, "128K":131072, "256K":262144}
    if model not in allowed or model not in _MODELS or context not in contexts:
        raise ValueError('Please choose from the provided model and context options.')
    if not isinstance(mmproj, bool):
        raise ValueError("MMProj parameter must be boolean.")
    previous = (MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED)
    MODEL, LOAD_MMPROJ, CONTEXT = model, mmproj, context
    _CTX, _SELECTED = contexts[context], _MODELS[model]
    print(f'Preparing model: {MODEL} | MMProj: {mmproj} | Context: {_CTX}', flush=True)
    try:
        for path in (_ROOT, _ROOT/"downloads", _ROOT/"logs", _ROOT/"metadata"):
            path.mkdir(parents=True, exist_ok=True)
        print('(1) Preparing new model; keeping current service online.', flush=True)
        info = _ensure_llama()
        paths = _prepare_files()
    except BaseException:
        MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED = previous
        print('Preparation failed or cancelled; previous service untouched and settings restored.', flush=True)
        raise
    print('(2) Files ready. Stopping old service and loading new model.', flush=True)
    stop_model_service()
    time.sleep(1)
    server_process = None
    try:
        server_process = _start_model(info, paths)
    except BaseException:
        print('Loading failed; old service stopped, model files preserved. Please retry with previous model or smaller context.', flush=True)
        raise
    if _is_suma_alive():
        print('(3) New model API ready; SUMA AI Bridge remains running without reconnecting.', flush=True)
    else:
        print('(3) New model API ready; SUMA not detected, launching SUMA automatically.', flush=True)
        try:
            _start_suma()
        except BaseException:
            print('SUMA launch incomplete; model server is running, check logs.', flush=True)
            raise
    list_downloaded_models()
    print(f"Switched: {MODEL}\nAPI: http://127.0.0.1:{_PORT}/v1\nModel alias: {_SELECTED['alias']}", flush=True)
    print('Please verify SUMA pairing / remote connection status with its live output.', flush=True)
    return server_process

server_process = download_and_switch(NEW_MODEL, MMPROJ, CONTEXT_SIZE)


In [ ]:
# @title 🔍 Service Health Check & Reconnect
# @markdown Run this cell after reconnecting to check files and service status, and automatically restart if needed.

import json
import os
import signal
import socket
import subprocess
import time
import urllib.request
from pathlib import Path

_ROOT = Path('/content/llama-direct-b12651') if Path('/content').exists() else Path.home()/'llama-direct-b12651'
_MODELS_ROOT = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
_PORT = 8081

print("=" * 60)
print('🔍 Checking Colab runtime and downloaded files status...')
print("=" * 60)

# 1. Check whether root directory and downloaded models were cleared
models_dir = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
packages_dir = _ROOT / 'packages'
existing_models = list(models_dir.glob('**/*.gguf')) if models_dir.exists() else []

if not _ROOT.exists() or (not existing_models and not packages_dir.exists()):
    print('❌ [Warning] Temporary files and downloaded models were cleared by Colab!')
    print('   Possible cause: The Colab runtime disconnected completely or the GPU was reclaimed.')
    print('👉 Please re-run the [Launch Form] cell above to initialize and download again.')
else:
    print(f'✅ Filesystem intact: Found {len(existing_models)} GGUF model file(s).')
    for m in existing_models:
        size_gb = m.stat().st_size / (1024 ** 3)
        print(f"   - {m.name} ({size_gb:.2f} GB)")

    # 2. Check llama-server service status
    def check_server_status():
        try:
            req = urllib.request.Request(f'http://127.0.0.1:{_PORT}/health')
            opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
            with opener.open(req, timeout=3) as resp:
                data = json.load(resp)
                return data.get('status') == 'ok'
        except Exception:
            return False

    server_ok = check_server_status()
    if server_ok:
        print(f'✅ llama.cpp model server is running normally (http://127.0.0.1:{_PORT}/v1).')
    else:
        print('⚠️ llama.cpp server is currently not responding.')

    # 3. Check SUMA AI Bridge
    suma_proc_file = _ROOT / 'suma-process.json'
    suma_alive = False
    if suma_proc_file.exists():
        try:
            saved = json.loads(suma_proc_file.read_text())
            pid = saved.get('pid')
            if pid:
                os.kill(pid, 0)
                suma_alive = True
        except (OSError, Exception):
            suma_alive = False

    if suma_alive:
        print('✅ SUMA AI Bridge process is running normally.')
    else:
        print('⚠️ SUMA AI Bridge is not running or has stopped.')

    # 4. Automatically enter Keep-Alive mode upon confirmation or restart to prevent idle timeout
    if server_ok and suma_alive:
        print('\n🎉 All services are running normally!')
        print('🛡️ Entering Keep-Alive mode automatically to prevent Colab idle disconnects...')
        import datetime, time
        cycle = 0
        try:
            while True:
                time.sleep(300)
                cycle += 5
                now_str = datetime.datetime.now().strftime("%H:%M:%S")
                print(f'[{now_str}] 💓 Keep-Alive Heartbeat: Running for {cycle} mins | Connection & GPU active', flush=True)
        except KeyboardInterrupt:
            print('\n🛑 Keep-Alive watchdog stopped manually.', flush=True)
    else:
        print('\n🔄 Attempting to restart interrupted services...')
        try:
            if 'info' in globals() and 'paths' in globals() and '_start_model' in globals():
                if not server_ok:
                    print('  - Restarting model server...')
                    server_process = _start_model(info, paths)
                if not suma_alive and '_start_suma' in globals():
                    print('  - Restarting SUMA AI Bridge...')
                    _start_suma()
                print('✅ Services restarted successfully!')
                print('🛡️ Entering Keep-Alive mode automatically to prevent Colab idle disconnects...')
                import datetime, time
                cycle = 0
                try:
                    while True:
                        time.sleep(300)
                        cycle += 5
                        now_str = datetime.datetime.now().strftime("%H:%M:%S")
                        print(f'[{now_str}] 💓 Keep-Alive Heartbeat: Running for {cycle} mins | Connection & GPU active', flush=True)
                except KeyboardInterrupt:
                    print('\n🛑 Keep-Alive watchdog stopped manually.', flush=True)
            else:
                print('ℹ️ Runtime variables were reset, but downloaded files are still preserved on disk.')
                print('👉 Please re-run the [Launch Form] cell above; cached models and binaries will be reused without re-downloading!')
        except Exception as e:
            print(f'❌ Automatic restart failed: {e}')
            print('👉 Please re-run the [Launch Form] cell above; completed downloads will be reused automatically.')
